In [20]:
import tensorflow as tf
import numpy as np
import evaluate
from transformers import AutoTokenizer, DataCollatorWithPadding, TFAutoModelForSequenceClassification
from datasets import load_dataset
from transformers.keras_callbacks import KerasMetricCallback
from transformers import TextClassificationPipeline

In [21]:
model = "prajjwal1/bert-tiny"
tokenizer = AutoTokenizer.from_pretrained(model, model_max_length=512)

In [22]:
def preprocessing_function(examples):
    return tokenizer(examples['text'], truncation=True)

In [23]:
imdb = load_dataset("imdb")

In [24]:
imdb.column_names

{'train': ['text', 'label'],
 'test': ['text', 'label'],
 'unsupervised': ['text', 'label']}

In [25]:
tokenized_imdb = imdb.map(preprocessing_function, batched=True)

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [26]:
small_tr = tokenized_imdb['train'].shuffle(seed=42).select(range(7500))
small_ts = tokenized_imdb['test'].shuffle(seed=42).select(range(5000))

In [27]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors='tf') # use 'tf' flag to train the model with HF using tensorflow

In [28]:
data_collator

DataCollatorWithPadding(tokenizer=BertTokenizerFast(name_or_path='prajjwal1/bert-tiny', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}, padding=True, max_length=None, pad_to_multiple_of=None, return_tensors='tf')

In [29]:
id2label = {0: 'NEGATIVE', 1: 'POSITIVE'}
label2id = {'NEGATIVE': 0, 'POSITIVE': 1}

In [30]:
tf_model = TFAutoModelForSequenceClassification.from_pretrained(
    model, 
    num_labels=2, 
    id2label=id2label, 
    label2id=label2id,
    from_pt=True
) # gets the pytorch pre-trained weights from HF and converts them for tensorflow

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertForSequenceClassification: ['bert.embeddings.position_ids']
- This IS expected if you are initializing TFBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [31]:
tf_train_set = tf_model.prepare_tf_dataset(
    small_tr,
    shuffle=True,
    batch_size=32,
    collate_fn=data_collator
)

tf_validation_set = tf_model.prepare_tf_dataset(
    small_ts,
    shuffle=False,
    batch_size=32,
    collate_fn=data_collator
)

In [32]:
tf_model.compile(optimizer=tf.keras.optimizers.legacy.Adam(learning_rate=5e-5))

In [33]:
accuracy = evaluate.load('accuracy')

In [34]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [35]:
metric_callback = KerasMetricCallback(metric_fn=compute_metrics, eval_dataset=tf_validation_set)

In [36]:
tf_model.fit(
    tf_train_set,
    validation_data=tf_validation_set,
    epochs=3, 
    callbacks=[metric_callback]
)

Epoch 1/3
234/234 [==============================] - 161s 683ms/step - loss: 0.6182 - val_loss: 0.4805 - accuracy: 0.7824
Epoch 2/3
234/234 [==============================] - 187s 800ms/step - loss: 0.3999 - val_loss: 0.3712 - accuracy: 0.8358
Epoch 3/3
234/234 [==============================] - 188s 803ms/step - loss: 0.2960 - val_loss: 0.3879 - accuracy: 0.8370


In [37]:
# Positive — clear enthusiasm, no hedging
positive_txt = ["""This film is an absolute triumph. From the 
opening scene to the final frame, every performance 
is magnetic, the script is witty and moving in equal 
measure, and the direction is confident without ever 
feeling showy. I laughed, I cried, and I left the 
cinema wanting to tell everyone I know to go and see 
it immediately. One of the best films I've watched 
in years — I can't recommend it highly enough."""]

# Negative — straightforward disappointment
negative_txt = ["""A complete waste of time. The plot 
makes no sense, the characters are flat and 
unlikeable, and the dialogue is cringe-worthy from 
start to finish. The pacing is so slow I checked my 
watch three times in the first hour. Nothing works — 
not the acting, not the direction, not the score. 
I genuinely regret buying a ticket and would 
not wish this film on anyone."""]

# Neutral — balanced, neither positive nor negative
neutral_txt = ["""It's a decent enough film. There are 
moments that genuinely work — a handful of good 
performances and a couple of well-crafted scenes — 
but there are just as many that fall flat. It's the 
kind of film you watch on a rainy afternoon and 
forget about by evening. Not bad, not good; perfectly 
average. Worth a watch if you have nothing else on, 
but don't go out of your way for it."""]

classifier = TextClassificationPipeline(tokenizer=tokenizer, model=tf_model, return_all_scores=True)

for label, txt in [("Positive", positive_txt), ("Negative", negative_txt), ("Neutral", neutral_txt)]:
    result = classifier(txt)[0]
    best = max(result, key=lambda x: x['score'])
    print(f"{label} review — model predicted: {best['label']} ({best['score']:.4f})")

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


Positive review — model predicted: POSITIVE (0.9644)
Negative review — model predicted: NEGATIVE (0.9681)
Neutral review — model predicted: NEGATIVE (0.8955)


/Users/antonioponti/miniconda3/envs/mlenv/lib/python3.11/site-packages/transformers/pipelines/text_classification.py:106: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(
